# Step 2 · 2주차 — 전처리와 특징 추출
컴퓨터소프트웨어학부 22114553 전진성 · https://github.com/jjs523/semg-gesture-angle

원신호를 윈도우당 25차원 특징 벡터로 바꾸고, 시행 단위로 먼저 나눈 데이터셋 `data/dataset.npz`를 만든다. 실행: `python main.py week2` (약 4분) 후 이 노트북.

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np, pandas as pd
from IPython.display import Image, display
import common as C, features as F
X, meta = C.load_all()
print('시행 배열', X.shape)

## 단계 1 · 필터 적용 — 60 Hz 노치 + 20~450 Hz 4차 Butterworth, `filtfilt(axis=0)`

In [ ]:
Xf = F.preprocess(X.astype(np.float64)).astype(np.float32)
i0 = meta.index[(meta.participant=='a')&(meta.label=='30_fist')&(meta.trial==0)][0]
print('전:', X[i0].std(0).round(4))
print('후:', Xf[i0].std(0).round(4))
display(Image('results/week2/filter_check.png'))

## 단계 2 · 분석 구간 — mark + 1.0~3.0 s (1주차에 측정한 활성 구간 0.72~3.22 s 근거, 재현 가정)

In [ ]:
seg = F.cut(Xf[i0]); print('세그먼트 모양:', seg.shape)
display(Image('results/week2/segment_windows.png'))

## 단계 3 · 슬라이딩 윈도우 — 200 ms, 100 ms 간격

In [ ]:
w = F.make_windows(seg); print('윈도우 배열:', w.shape, '  (2000-200)/100+1 =', (2000-200)//100+1)

## 단계 4 · 시간영역 특징 — 논문 식 (1)~(5), ZC만 임계값 0.01

In [ ]:
import inspect; print(inspect.getsource(F.td_features))
f = F.td_features(w[0]); print('특징 벡터:', f.shape)
print(dict(zip(F.FEATURE_NAMES, np.round(f, 4))))

In [ ]:
display(Image('results/week2/feature_class_heatmap.png'))

## 단계 5 · 데이터셋 — **시행 단위로 먼저 나눈 뒤** 각 집합 안에서만 윈도우·특징 생성

In [ ]:
from sklearn.model_selection import train_test_split
strat = (meta.label + '_' + meta.participant).values          # 클래스 x 피험자 층화
tr_idx, te_idx = train_test_split(np.arange(len(meta)), test_size=0.2, stratify=strat, random_state=42)  # 1) 시행 단위 분할
assert not set(tr_idx) & set(te_idx)
Xtr = F.trial_features(Xf[np.sort(tr_idx)]).reshape(-1, 25)   # 2) 학습 시행 안에서만 특징
Xte = F.trial_features(Xf[np.sort(te_idx)]).reshape(-1, 25)   #    테스트 시행 안에서만 특징
print(Xtr.shape, Xte.shape, '비율', len(Xtr)/len(Xte))
pd.read_csv('results/week2/dataset_summary.csv', index_col=0)

## 단계 6 · 표준화와 저장 — StandardScaler 를 학습셋에만 fit

In [ ]:
d = np.load('data/dataset.npz', allow_pickle=True)
print({k: d[k].shape for k in d.files})
print('학습 평균 |최대|', abs(d['Xtr'].mean(0)).max().round(6), ' 학습 표준편차', d['Xtr'].std(0).min().round(4), '~', d['Xtr'].std(0).max().round(4))
print('테스트 평균 |최대|', abs(d['Xte'].mean(0)).max().round(3), ' 테스트 표준편차', d['Xte'].std(0).min().round(3), '~', d['Xte'].std(0).max().round(3))

## 설정 근거와 추가 검증 (학습 1,536시행 안 5-fold, 테스트셋은 보지 않음)

In [ ]:
display(Image('results/week2/settings_check.png'))
display(pd.read_csv('results/week2/window_length.csv'))
display(pd.read_csv('results/week2/leakage.csv'))

In [ ]:
display(pd.read_csv('results/week2/sample_unit.csv'))
display(Image('results/week2/confound.png'))
display(pd.read_csv('results/week2/rest_confound.csv'))

**정리**: 논문 수준(96 %)은 윈도우 하나가 아니라 세그먼트(19윈도우) 단위에서 나온다. 손을 쉬는 4~5 s 구간만으로도 손동작을 87 % 맞히므로, 같은 녹음 파일의 시행이 학습·평가에 섞이는 trial-wise 평가에는 녹음 세션 교란이 들어 있다. 3주차에 피험자 단위(LOSO) 평가로 이를 분리한다.